In [1]:
import tensorflow as tf
print('GPU Available:', tf.config.list_physical_devices('GPU'))
print('TensorFlow:', tf.__version__)

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TensorFlow: 2.20.0


In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_FOLDER = '/content/drive/MyDrive/MarketPredictor'
os.makedirs(PROJECT_FOLDER, exist_ok=True)
os.chdir(PROJECT_FOLDER)
print('Working directory:', os.getcwd())
print('All files will be saved here!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/MarketPredictor
All files will be saved here!


In [3]:
!pip install -q yfinance praw vaderSentiment feedparser pandas numpy matplotlib seaborn scikit-learn tensorflow wordcloud requests

In [4]:
import os, re, warnings, datetime, pickle
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import yfinance as yf
import feedparser
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                              mean_squared_error, classification_report)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (SimpleRNN, LSTM, GRU, Dense, Dropout,
                                      Bidirectional, BatchNormalization, Input)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

np.random.seed(42)
tf.random.set_seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print('Ready | TF:', tf.__version__)

Ready | TF: 2.20.0


In [5]:
PRIMARY_TICKER  = 'AAPL'
TICKERS         = ['AAPL', 'GOOGL', 'MSFT', 'AMZN', 'TSLA']
START_DATE      = '2020-01-01'
END_DATE        = '2024-12-31'
SEQUENCE_LENGTH = 30
EPOCHS          = 100
BATCH_SIZE      = 16
PATIENCE        = 20
REDDIT_CLIENT_ID     = 'YOUR_CLIENT_ID'
REDDIT_CLIENT_SECRET = 'YOUR_CLIENT_SECRET'
REDDIT_USER_AGENT    = 'MarketPredictor/1.0'

RSS_FEEDS = [
    'https://feeds.reuters.com/reuters/businessNews',
    'https://feeds.reuters.com/reuters/technologyNews',
]
print('Config ready |', PRIMARY_TICKER, '|', START_DATE, 'to', END_DATE)

Config ready | AAPL | 2020-01-01 to 2024-12-31


In [6]:
print('Downloading stock data.')
all_stocks = {}
for t in TICKERS:
    df = yf.download(t, start=START_DATE, end=END_DATE, progress=False)
    df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
    all_stocks[t] = df
    print(f'  {t}: {len(df)} rows')

df_raw = all_stocks[PRIMARY_TICKER].copy().fillna(method='ffill').dropna()

def add_indicators(df):
    df = df.copy()
    df['Returns']       = df['Close'].pct_change()
    df['MA_5']          = df['Close'].rolling(5).mean()
    df['MA_20']         = df['Close'].rolling(20).mean()
    df['MA_50']         = df['Close'].rolling(50).mean()
    df['EMA_12']        = df['Close'].ewm(span=12).mean()
    df['EMA_26']        = df['Close'].ewm(span=26).mean()
    df['MACD']          = df['EMA_12'] - df['EMA_26']
    df['MACD_Signal']   = df['MACD'].ewm(span=9).mean()
    delta = df['Close'].diff()
    gain  = delta.where(delta > 0, 0).rolling(14).mean()
    loss  = (-delta.where(delta < 0, 0)).rolling(14).mean()
    df['RSI']           = 100 - (100 / (1 + gain / (loss + 1e-10)))
    df['BB_Mid']        = df['Close'].rolling(20).mean()
    df['BB_Std']        = df['Close'].rolling(20).std()
    df['BB_Upper']      = df['BB_Mid'] + 2 * df['BB_Std']
    df['BB_Lower']      = df['BB_Mid'] - 2 * df['BB_Std']
    df['BB_Width']      = (df['BB_Upper'] - df['BB_Lower']) / (df['BB_Mid'] + 1e-10)
    df['Volatility_5']  = df['Returns'].rolling(5).std()
    df['Volatility_20'] = df['Returns'].rolling(20).std()
    df['Volume_MA_5']   = df['Volume'].rolling(5).mean()
    df['Volume_Ratio']  = df['Volume'] / (df['Volume_MA_5'] + 1e-10)
    df['Direction']     = (df['Returns'].shift(-1) > 0).astype(int)
    return df.dropna()

df_feat = add_indicators(df_raw)
print('\nFeature shape:', df_feat.shape)
print('Class balance — Up:', df_feat['Direction'].sum(), '| Down:', (df_feat['Direction']==0).sum())

  AAPL: 1257 rows
  GOOGL: 1257 rows
  MSFT: 1257 rows
  AMZN: 1257 rows
  TSLA: 1257 rows

Feature shape: (1208, 24)
Class balance — Up: 647 | Down: 561


In [8]:
analyzer = SentimentIntensityAnalyzer()
articles = []
# Reuters RSS
print('Fetching Reuters RSS...')
for url in RSS_FEEDS:
    try:
        feed = feedparser.parse(url)
        for e in feed.entries[:100]:
            try:
                date = pd.to_datetime(e.get('published', '')).date()
            except:
                date = datetime.date.today()
            text = e.get('title', '') + ' ' + e.get('summary', '')
            score = analyzer.polarity_scores(text)['compound']
            sentiment = 'Positive' if score > 0.05 else ('Negative' if score < -0.05 else 'Neutral')
            articles.append({'date': date, 'title': e.get('title', ''),
                              'summary': e.get('summary', ''), 'source': 'Reuters',
                              'sentiment': sentiment, 'sentiment_score': score,
                              'text': text})
        print(f'  RSS: {len(feed.entries)} articles')
    except Exception as ex:
        print(f'  RSS error: {ex}')
try:
    import praw
    r = praw.Reddit(client_id=REDDIT_CLIENT_ID,
                    client_secret=REDDIT_CLIENT_SECRET,
                    user_agent=REDDIT_USER_AGENT)
    for sub in ['investing', 'stocks', 'wallstreetbets']:
        try:
            for p in r.subreddit(sub).hot(limit=50):
                text  = p.title + ' ' + p.selftext[:300]
                score = analyzer.polarity_scores(text)['compound']
                sentiment = 'Positive' if score > 0.05 else ('Negative' if score < -0.05 else 'Neutral')
                articles.append({
                    'date': datetime.datetime.fromtimestamp(p.created_utc).date(),
                    'title': p.title, 'summary': p.selftext[:300],
                    'source': 'Reddit/r/' + sub,
                    'sentiment': sentiment, 'sentiment_score': score, 'text': text
                })
        except:
            pass
    print('Reddit collected')
except:
    print('Reddit not configured — skipping')
print('Adding simulated news for coverage.')
import random
random.seed(42)
pos_headlines = [
    f'{PRIMARY_TICKER} surges on strong earnings',
    f'Analysts upgrade {PRIMARY_TICKER} target',
    f'{PRIMARY_TICKER} beats revenue estimates',
    'Markets rally on positive economic data',
    'Tech stocks climb on strong outlook',
]
neg_headlines = [
    f'{PRIMARY_TICKER} falls on weak guidance',
    f'Investors concerned about {PRIMARY_TICKER} margins',
    'Markets drop amid recession fears',
    'Tech selloff continues on rate hike worries',
    f'{PRIMARY_TICKER} misses earnings expectations',
]
neu_headlines = [
    f'{PRIMARY_TICKER} announces product update',
    'Fed holds rates steady at latest meeting',
    f'{PRIMARY_TICKER} CFO speaks at conference',
    'Markets mixed ahead of jobs report',
]
date_range = pd.date_range(start=START_DATE, end=END_DATE, freq='B')
for date in date_range:
    for _ in range(random.randint(1, 3)):
        pool = pos_headlines + neg_headlines + neu_headlines
        title = random.choice(pool)
        score = analyzer.polarity_scores(title)['compound']
        sentiment = 'Positive' if score > 0.05 else ('Negative' if score < -0.05 else 'Neutral')
        articles.append({'date': date.date(), 'title': title, 'summary': '',
                          'source': 'Simulated', 'sentiment': sentiment,
                          'sentiment_score': score, 'text': title})

df_all_news = pd.DataFrame(articles)
df_all_news['date'] = pd.to_datetime(df_all_news['date'])
print(f'\nTotal news articles: {len(df_all_news)}')
print(df_all_news['sentiment'].value_counts())

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Fetching Reuters RSS...
  RSS: 0 articles
  RSS: 0 articles


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Reddit collected
Adding simulated news for coverage.

Total news articles: 2636
sentiment
Neutral     1317
Negative     770
Positive     549
Name: count, dtype: int64


In [9]:
daily_sent = df_all_news.groupby('date').agg(
    sentiment_mean  = ('sentiment_score', 'mean'),
    positive_ratio  = ('sentiment', lambda x: (x == 'Positive').mean()),
    negative_ratio  = ('sentiment', lambda x: (x == 'Negative').mean()),
    article_count   = ('sentiment_score', 'count')
).reset_index()
daily_sent['date'] = pd.to_datetime(daily_sent['date'])
daily_sent.set_index('date', inplace=True)
# Merge with price features
df_merged = df_feat.copy()
df_merged.index = pd.to_datetime(df_merged.index)
df_merged = df_merged.join(daily_sent, how='left')
df_merged['sentiment_mean'].fillna(0, inplace=True)
df_merged['positive_ratio'].fillna(0.33, inplace=True)
df_merged['negative_ratio'].fillna(0.33, inplace=True)
df_merged['article_count'].fillna(0, inplace=True)
df_merged.dropna(inplace=True)

print('Merged dataset shape:', df_merged.shape)

Merged dataset shape: (1208, 28)


In [10]:
FEATURE_COLS = [
    'Close', 'Volume', 'Returns', 'MA_5', 'MA_20', 'MA_50',
    'EMA_12', 'EMA_26', 'MACD', 'MACD_Signal', 'RSI',
    'BB_Width', 'BB_Upper', 'BB_Lower',
    'Volatility_5', 'Volatility_20', 'Volume_Ratio',
    'sentiment_mean', 'positive_ratio', 'negative_ratio'
]
# Keep only columns that exist
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_merged.columns]

X_raw = df_merged[FEATURE_COLS].values
y_all = df_merged['Direction'].values

# Scale
sc_fit = MinMaxScaler()
X_scaled = sc_fit.fit_transform(X_raw)

# Sliding window sequences
X_seq, y_seq = [], []
for i in range(SEQUENCE_LENGTH, len(X_scaled)):
    X_seq.append(X_scaled[i - SEQUENCE_LENGTH:i])
    y_seq.append(y_all[i])
X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

# 70 / 15 / 15 split  (no shuffle — time-series order must be kept)
n      = len(X_seq)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

X_train, y_train = X_seq[:n_train],         y_seq[:n_train]
X_val,   y_val   = X_seq[n_train:n_train+n_val], y_seq[n_train:n_train+n_val]
X_test,  y_test  = X_seq[n_train+n_val:],    y_seq[n_train+n_val:]

INPUT_SHAPE = (X_train.shape[1], X_train.shape[2])

# Class weights
classes          = np.unique(y_train)
cw               = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, cw))

print(f'Sequences  : {len(X_seq)}')
print(f'Train/Val/Test: {X_train.shape[0]} / {X_val.shape[0]} / {X_test.shape[0]}')
print(f'Input shape: {INPUT_SHAPE}')
print(f'Features   : {FEATURE_COLS}')
print(f'Class weights: {class_weight_dict}')
print(f'Train — Up: {y_train.sum()} | Down: {(y_train==0).sum()}')

Sequences  : 1178
Train/Val/Test: 824 / 176 / 178
Input shape: (30, 20)
Features   : ['Close', 'Volume', 'Returns', 'MA_5', 'MA_20', 'MA_50', 'EMA_12', 'EMA_26', 'MACD', 'MACD_Signal', 'RSI', 'BB_Width', 'BB_Upper', 'BB_Lower', 'Volatility_5', 'Volatility_20', 'Volume_Ratio', 'sentiment_mean', 'positive_ratio', 'negative_ratio']
Class weights: {np.int64(0): np.float64(1.0537084398976981), np.int64(1): np.float64(0.9515011547344111)}
Train — Up: 433 | Down: 391


In [35]:
# Run this ONE cell first
import pickle
fcols = pickle.load(open('feature_cols.pkl', 'rb'))
print(f'Total features: {len(fcols)}')
for i, f in enumerate(fcols):
    print(f'  {i+1}. {f}')

Total features: 20
  1. Close
  2. Volume
  3. Returns
  4. MA_5
  5. MA_20
  6. MA_50
  7. EMA_12
  8. EMA_26
  9. MACD
  10. MACD_Signal
  11. RSI
  12. BB_Width
  13. BB_Upper
  14. BB_Lower
  15. Volatility_5
  16. Volatility_20
  17. Volume_Ratio
  18. sentiment_mean
  19. positive_ratio
  20. negative_ratio


In [37]:
import gradio as gr
import numpy as np
import pickle
import yfinance as yf
from tensorflow.keras.models import load_model
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Load model and files
model    = load_model('best_model.h5')
scaler   = pickle.load(open('scaler.pkl', 'rb'))
fcols    = pickle.load(open('feature_cols.pkl', 'rb'))
analyzer = SentimentIntensityAnalyzer()

def predict_market(ticker, headline):
    try:
        # Download stock data
        df = yf.download(ticker.strip().upper(), period='90d', progress=False)
        if len(df) < 60:
            return "Error: Not enough data. Try valid ticker like AAPL."

        # Fix column names
        df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]

        # ── All 20 features exactly as training ──
        df['Returns']       = df['Close'].pct_change()
        df['MA_5']          = df['Close'].rolling(5).mean()
        df['MA_20']         = df['Close'].rolling(20).mean()
        df['MA_50']         = df['Close'].rolling(50).mean()
        df['EMA_12']        = df['Close'].ewm(span=12).mean()
        df['EMA_26']        = df['Close'].ewm(span=26).mean()
        df['MACD']          = df['EMA_12'] - df['EMA_26']
        df['MACD_Signal']   = df['MACD'].ewm(span=9).mean()
        delta               = df['Close'].diff()
        gain                = delta.where(delta>0, 0).rolling(14).mean()
        loss                = (-delta.where(delta<0, 0)).rolling(14).mean()
        df['RSI']           = 100 - (100/(1+gain/(loss+1e-10)))
        df['BB_Mid']        = df['Close'].rolling(20).mean()
        df['BB_Std']        = df['Close'].rolling(20).std()
        df['BB_Upper']      = df['BB_Mid'] + 2*df['BB_Std']
        df['BB_Lower']      = df['BB_Mid'] - 2*df['BB_Std']
        df['BB_Width']      = (df['BB_Upper']-df['BB_Lower'])/(df['BB_Mid']+1e-10)
        df['Volatility_5']  = df['Returns'].rolling(5).std()
        df['Volatility_20'] = df['Returns'].rolling(20).std()
        df['Volume_MA_5']   = df['Volume'].rolling(5).mean()
        df['Volume_Ratio']  = df['Volume']/(df['Volume_MA_5']+1e-10)

        # Sentiment
        sc = analyzer.polarity_scores(headline)['compound'] if headline.strip() else 0.0
        df['sentiment_mean']  = sc
        df['positive_ratio']  = 1.0 if sc >  0.05 else 0.0
        df['negative_ratio']  = 1.0 if sc < -0.05 else 0.0

        df = df.dropna()

        if len(df) < 30:
            return "Error: Not enough data after processing. Try longer period."

        # Exact 20 features in exact order
        avail = [c for c in fcols if c in df.columns]

        if len(avail) != len(fcols):
            missing = [c for c in fcols if c not in df.columns]
            return f"Error: Missing features: {missing}"

        # Last 30 rows → shape (1, 30, 20)
        X = scaler.transform(df[avail].values[-30:])
        X = X.reshape(1, 30, len(avail))

        # Predict
        prob      = float(model.predict(X, verbose=0)[0][0])
        direction = 'UP ↑' if prob >= 0.5 else 'DOWN ↓'
        conf      = prob if prob >= 0.5 else 1 - prob
        sentiment = 'Positive ' if sc > 0.05 else 'Negative ' if sc < -0.05 else 'Neutral '

        return (
            f"Direction   : {direction}\n"
            f"Confidence  : {conf*100:.1f}%\n"
            f"Sentiment   : {sentiment}\n"
            f"Sent Score  : {sc:.3f}\n"
            f"Last Close  : ${float(df['Close'].iloc[-1]):.2f}"
        )

    except Exception as e:
        import traceback
        return f"Error: {str(e)}\n{traceback.format_exc()}"


# Gradio Interface
demo = gr.Interface(
    fn=predict_market,
    inputs=[
        gr.Textbox(
            label='Stock Ticker',
            value='AAPL',
            placeholder='e.g. AAPL, GOOGL, MSFT, TSLA, AMZN'
        ),
        gr.Textbox(
            label='News Headline (optional)',
            value='',
            placeholder='e.g. Apple reports record earnings beating expectations',
            lines=2
        )
    ],
    outputs=gr.Textbox(label='Prediction', lines=6),
    title=' Market Movement Predictor',
    description='Enter stock ticker and optional news headline to predict next day direction',
    examples=[
        ['AAPL', 'Apple reports record quarterly earnings beating analyst expectations'],
        ['GOOGL', 'Google faces antitrust lawsuit as regulators crack down on tech giants'],
        ['TSLA', 'Tesla deliveries miss targets amid production challenges'],
        ['MSFT', 'Microsoft Azure cloud revenue surges 30 percent year over year'],
        ['AMZN', ''],
    ],
    theme=gr.themes.Soft()
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://abfa8b54697cedc420.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
